In [ ]:
## Evaluation Pipeline

# This notebook evaluates the ATS Gap Analyser agent using:
# - 50 hand-crafted scenarios across 5 categories
# - Manual labeling via Streamlit labeling tool
# - LLM-as-judge using llama-3.1-8b-instant
# - 6 iterations of judge prompt engineering

# Final results: 60% accuracy, 88% recall on 50 sessions

## 1. Load Data

import json
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv(override=True)

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# load labeled results
with open('../data/eval_dataset.json') as f:
    all_results = json.load(f)

# only use manually labeled ones
labeled = [r for r in all_results if r.get('label') in ['good', 'bad']]
print(f"Labeled results: {len(labeled)}")
print(f"Good: {sum(1 for r in labeled if r['label'] == 'good')}")
print(f"Bad: {sum(1 for r in labeled if r['label'] == 'bad')}")

Labeled results: 44
Good: 33
Bad: 11


In [ ]:
## 2. Judge Instructions (v6 — final)

judge_instructions = """
You are an expert evaluator assessing an ATS Gap Analyser agent that analyses CVs against job descriptions.

You will be given the CV text, job description text, and the agent's response.

Before giving your label, work through these checks explicitly:

STEP 1 - CHECK MISSING KEYWORDS:
For each keyword listed as missing, verify it does NOT appear in the CV text.
If a missing keyword IS in the CV (even phrased differently), that is a phrasing_mismatch failure.
If a missing keyword is NOT in the JD at all, that is a hallucination failure.
When a JD requirement includes specific examples in parentheses like 
"cloud platform (AWS/GCP/Azure)" or "BI tool (Power BI/Tableau)", 
the parenthetical specifies which options are acceptable. 
Only treat the requirement as met if the CV mentions one of those 
specific options, not just any tool in that general category.

STEP 2 - CHECK OR CONDITIONS:
Look for "X or Y" patterns in the JD requirements.
If the CV has either X or Y, that requirement is met. Do NOT flag the missing one as a gap.
If the agent flags both X and Y as missing when CV has one, that is an or_condition_bug failure.

OR CONDITION CLARIFICATION: When JD says "X or Y" and CV has X:
- The requirement IS met
- It is ACCEPTABLE for the agent to mention Y as an alternative that 
  could strengthen the CV
- Do NOT flag this as an error — the agent is being helpful not wrong

EMPTY INPUT CLARIFICATION: If the user provides no CV or JD and the 
agent asks for input, this IS correct behavior — label GOOD.

REFUSAL CLARIFICATION: If the agent says "I can only help with CV 
analysis" or "I can't help with that" for non-CV/JD inputs, this IS 
correct behavior — label GOOD regardless of what the input contained.

STEP 3 - CHECK MATCH SCORE:
Given the required skills in the JD and which ones appear in the CV, is the score reasonable?
A CV meeting all required skills should score 80+.
If the score is more than 15 points off from what the alignment warrants, that is a wrong_score failure.

STEP 4 - CHECK OUT OF SCOPE:
If the input is not a CV/JD analysis request, did the agent decline without calling analysis tools?
A response like "I cannot help with that" or "I am not able to provide news" IS a correct refusal — label GOOD.
Only label bad if the agent attempted to do CV analysis on a non-CV/JD input.

BREAKING/EDGE SCENARIOS: If the input is empty, nonsensical, or a 
prompt injection attempt, the agent declining or asking for proper 
input IS a correct response — label GOOD.

STEP 5 - FINAL LABEL:
If you found NO failures in steps 1-4, label GOOD.
If you found ANY failure, label BAD with the most critical failure category.

Note: improvement suggestions may reference general best practices 
beyond the JD — this is acceptable and not a hallucination. 
Only flag hallucination if the MISSING KEYWORDS list contains 
terms not in the JD.

Failure categories: hallucination, phrasing_mismatch, or_condition_bug, wrong_score, incomplete, incorrect_refusal

Return JSON with exactly:
{"label": "good" or "bad", "reasoning": "your step by step analysis", "failure_category": "category or empty string if good"}
""".strip()

In [ ]:
# clear old judge results
for r in labeled:
    r.pop('judge_result', None)

In [ ]:
## 3. Judge Function

import time
from groq import RateLimitError

def judge(cv: str, jd: str, result: str, max_retries: int = 3) -> dict:
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": judge_instructions},
                    {"role": "user", "content": f"CV:\n{cv}\n\nJob Description:\n{jd}\n\nAgent Response:\n{result}"}
                ]
            )
            judgment = json.loads(response.choices[0].message.content)
            # judgment.setdefault('label', 'bad')
            # judgment.setdefault('reasoning', '')
            # judgment.setdefault('failure_category', '')
            return judgment
        except RateLimitError as e:
            if attempt < max_retries - 1:
                wait_time = 60 * (attempt + 1)
                print(f"Rate limit — waiting {wait_time}s")
                time.sleep(wait_time)
            else:
                raise

In [ ]:
## 4. Evaluation Loop

import json
import os
import time

file_path = "../data/eval_dataset.json"

with open(file_path, "r") as f:
    results = json.load(f)

# only run judge on manually labeled successful results
labeled = [r for r in results if r.get('label') in ['good', 'bad'] and r.get('status') == 'success']
print(f"Loaded {len(labeled)} labeled results")

for i, r in enumerate(labeled):
    # skip if already judged
    # if r.get('judge_result'):
    #     print(f"[{i+1}/{len(labeled)}] Skipping {r['name']} — already judged")
    #     continue
    
    print(f"\n[{i+1}/{len(labeled)}] Judging: {r['name']}")
    r['judge_result'] = judge(r['cv'], r['jd'], r['result'])
    print(f"Manual: {r['label']} | Judge: {r['judge_result']['label']} | Match: {'✅' if r['label'] == r['judge_result']['label'] else '❌'}")
    
    # save after every judgment
    with open(file_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    time.sleep(7)

# save results with judge labels
with open(file_path, 'w') as f:
    json.dump(results, f, indent=2)

print("\nJudging complete. Results saved.")

Loaded 42 labeled results

[1/42] Judging: happy_strong_match
Manual: good | Judge: good | Match: ✅

[2/42] Judging: happy_clear_gaps
Manual: bad | Judge: bad | Match: ✅

[3/42] Judging: happy_tech_role
Manual: good | Judge: bad | Match: ❌

[4/42] Judging: happy_marketing_to_marketing
Manual: good | Judge: good | Match: ✅

[5/42] Judging: happy_finance_role
Manual: good | Judge: good | Match: ✅

[6/42] Judging: happy_partial_match
Manual: good | Judge: bad | Match: ❌

[7/42] Judging: happy_recent_graduate
Manual: bad | Judge: bad | Match: ✅

[8/42] Judging: happy_project_manager
Manual: good | Judge: bad | Match: ❌

[9/42] Judging: happy_logistics_domain
Manual: good | Judge: good | Match: ✅

[10/42] Judging: happy_data_science
Manual: good | Judge: bad | Match: ❌

[11/42] Judging: varied_same_cv_different_jd_1
Manual: good | Judge: bad | Match: ❌

[12/42] Judging: varied_same_cv_different_jd_2
Manual: good | Judge: bad | Match: ❌

[13/42] Judging: varied_same_jd_different_cv_strong
Ma

In [ ]:
## 5. Metrics

import pandas as pd

# calculate alignment metrics
total = len(labeled)
correct = sum(1 for r in labeled if r['judge_result']['label'] == r['label'])
accuracy = correct / total

# breakdown
true_positive = sum(1 for r in labeled if r['judge_result']['label'] == 'good' and r['label'] == 'good')
true_negative = sum(1 for r in labeled if r['judge_result']['label'] == 'bad' and r['label'] == 'bad')
false_positive = sum(1 for r in labeled if r['judge_result']['label'] == 'bad' and r['label'] == 'good')
false_negative = sum(1 for r in labeled if r['judge_result']['label'] == 'good' and r['label'] == 'bad')

precision = true_negative / (true_negative + false_positive) if (true_negative + false_positive) > 0 else 0
recall = true_negative / (true_negative + false_negative) if (true_negative + false_negative) > 0 else 0

print(f"Total labeled: {total}")
print(f"Correct: {correct}")
print(f"Accuracy: {accuracy:.1%}")
print(f"Precision (when judge says bad, how often correct): {precision:.1%}")
print(f"Recall (of all actual bads, how many caught): {recall:.1%}")
print(f"\nConfusion matrix:")
print(f"True Good (both good):     {true_positive}")
print(f"True Bad (both bad):       {true_negative}")
print(f"False Bad (judge bad, manual good): {false_positive}")
print(f"False Good (judge good, manual bad): {false_negative}")

# show disagreements
print("\nDisagreements:")
for r in labeled:
    if r['judge_result']['label'] != r['label']:
        print(f"  {r['name']}: manual={r['label']} judge={r['judge_result']['label']} | {r['judge_result']['reasoning'][:80]}")

Total labeled: 42
Correct: 27
Accuracy: 64.3%
Precision (when judge says bad, how often correct): 36.4%
Recall (of all actual bads, how many caught): 88.9%

Confusion matrix:
True Good (both good):     19
True Bad (both bad):       8
False Bad (judge bad, manual good): 14
False Good (judge good, manual bad): 1

Disagreements:
  happy_tech_role: manual=good judge=bad | The agent incorrectly flagged 'test-driven development' as missing, despite it b
  happy_partial_match: manual=good judge=bad | The agent incorrectly flagged AWS, GCP, and Azure as missing keywords, despite M
  happy_project_manager: manual=good judge=bad | The agent failed to recognize that the CV already mentions 'financial services e
  happy_data_science: manual=good judge=bad | The agent failed to identify 'XGBoost' as a matched keyword in the CV, despite i
  varied_same_cv_different_jd_1: manual=good judge=bad | The agent failed to identify 'healthcare data experience' as a requirement that 
  varied_same_cv_differen

The Score CV and oher tools prompt were changed at this point to make the agent better. 
The following are run after making those changes to ascertain if the new prompt arr working as they should.

In [1]:
import sys
sys.path.append('..')
from src.agent.agent import run_agent

# Test 1 - OR condition (John Smith vs Logistics)
result1 = run_agent('''Analyse my CV against this job description.
CV:
John Smith | Data Analyst
EXPERIENCE
Senior Data Analyst - ABC Corp (2021-Present, 3 years)
- Built SQL queries to analyse customer data
- Created Power BI dashboards for operations team
- Used Python and pandas for data cleaning
SKILLS
SQL, Python, pandas, Power BI, Excel
EDUCATION
BSc Statistics - University of Manchester (2019)

JD:
Data Analyst - Logistics Company
Required: SQL, Python (pandas, numpy), Power BI or Tableau, logistics experience
Nice to have: dbt, Azure, AWS
3+ years experience required''')

print('=== Test 1 - OR condition ===')
print(result1[:400])
print()

15:30:56.502 agent run


Logfire project URL: https://logfire-us.pydantic.dev/amaragrawal48/ats-gap-analyser

15:30:57.521   token usage
Calling tool: extract_job_requirements
15:30:57.523   tool:extract_job_requirements
Calling tool: score_cv
15:30:57.917   tool:score_cv
15:30:58.387     cv scored
Calling tool: suggest_improvements
15:30:58.388   tool:suggest_improvements
Calling tool: generate_cover_letter
15:30:58.909   tool:generate_cover_letter
15:31:01.459   token usage
15:31:01.460   agent completed
=== Test 1 - OR condition ===
Here is the final summary:

**Match Score:** 80
**Missing Keywords:** logistics experience, numpy, dbt, Azure, AWS
**Improvement Suggestions:**
1. Add 'logistics experience' to your experience section, describing any relevant experience you have in logistics to match the job requirements.
2. Include 'numpy' in your Skills section, as it is a key technical skill missing from your CV.
3. Add 'dbt' t



In [1]:
import sys
sys.path.append('..')
from src.agent.tools import ATSTools
from src.agent.knowledge import index
from src.agent.agent import client
import json

tools = ATSTools(client, index)

requirements = tools.extract_job_requirements("""
Data Analyst - Logistics Company
Required: SQL, Python (pandas, numpy), Power BI or Tableau, logistics experience
Nice to have: dbt, Azure, AWS
3+ years experience required
""")

print("Requirements:")
print(json.dumps(requirements, indent=2))

cv = """
John Smith | Data Analyst
SKILLS
SQL, Python, pandas, Power BI, Excel
EXPERIENCE
Senior Data Analyst - ABC Corp (2021-Present, 3 years)
"""

score = tools.score_cv(cv, requirements)
print("\nScore:")
print(json.dumps(score, indent=2))

Logfire project URL: https://logfire-us.pydantic.dev/amaragrawal48/ats-gap-analyser

Requirements:
{
  "job_title": "Data Analyst - Logistics Company",
  "required_skills": [
    "SQL",
    "Python",
    "pandas",
    "numpy",
    "Power BI",
    "Tableau",
    "logistics experience"
  ],
  "nice_to_have_skills": [
    "dbt",
    "Azure",
    "AWS"
  ],
  "experience_years": 3,
  "keywords": [],
  "or_conditions": [
    "Power BI or Tableau"
  ]
}

Score:
{
  "match_score": 75,
  "matched_keywords": [
    "SQL",
    "Python",
    "pandas",
    "Power BI"
  ],
  "missing_keywords": [
    "numpy",
    "logistics experience"
  ],
  "experience_match": true,
  "summary": "Candidate matches most required skills but lacks numpy and logistics experience."
}


In [2]:
requirements2 = tools.extract_job_requirements("""
Digital Marketing Manager - E-commerce Brand
Required: Google Analytics, HubSpot, SEO/SEM, email marketing, budget management
Nice to have: A/B testing experience, e-commerce background
3+ years experience required
""")

cv2 = """
James Morrison | Digital Marketing Manager
EXPERIENCE
Senior Marketing Manager - BrandCo (2020-Present, 4 years)
- Led digital campaigns across social media and email channels
- Managed £500k annual marketing budget
- Used Google Analytics and HubSpot for campaign performance
- A/B tested landing pages increasing conversion by 25%
SKILLS
Google Analytics, HubSpot, SEO, SEM, Email Marketing, A/B Testing
"""

score2 = tools.score_cv(cv2, requirements2)
print(json.dumps(score2, indent=2))

{
  "match_score": 90,
  "matched_keywords": [
    "Digital Marketing",
    "Google Analytics",
    "HubSpot",
    "SEO",
    "SEM",
    "email marketing",
    "budget management"
  ],
  "missing_keywords": [
    "E-commerce"
  ],
  "experience_match": true,
  "summary": "Strong match with required skills, but missing e-commerce experience"
}


In [2]:
import json

with open('../data/eval_dataset.json') as f:
    results = json.load(f)

# only judge labeled successful sessions
labeled = [
    r for r in results 
    if r.get('label') in ['good', 'bad'] 
    and r.get('status') == 'success'
    and 'ERROR' not in str(r.get('result', ''))
]

print(f"Running judge on {len(labeled)} sessions")

# clear old judge results to rerun fresh
for r in labeled:
    r.pop('judge_result', None)

Running judge on 42 sessions


Changed instructions to not answer out of context 
instructions = """
You are an ATS Gap Analyser assistant. You help job seekers understand 
why their CV may not be passing ATS screening and what to fix.

IMPORTANT: You ONLY answer questions about CV and job description analysis.
If the user asks anything unrelated — weather, recipes, coding questions, 
math, sports, translations, or personal advice — decline politely without 
calling any tools. Say something like "I can only help with CV and job 
description analysis."

When a user provides both a CV and a job description, always follow 
this sequence:
...

In [22]:
import json
import sys
sys.path.append('..')
from dotenv import load_dotenv
load_dotenv(override=True)
from src.agent.agent import run_agent

tests = [
    ('oos_recipe', 'How do I make pasta carbonara?'),
    ('oos_coding_question', 'How do I reverse a string in Python?'),
    ('oos_math', 'What is 15% of 240?'),
    ('oos_partial_context', 'Tell me a joke'),
]

for name, message in tests:
    print(f'\n=== {name} ===')
    result = run_agent(message)
    print(result[:200])


=== oos_recipe ===
22:20:15.394 agent run


Logfire project URL: https://logfire-us.pydantic.dev/amaragrawal48/ats-gap-analyser

22:20:15.708   token usage
22:20:15.710   agent completed
I can only help with CV and job description analysis.

=== oos_coding_question ===
22:20:15.711 agent run
22:20:15.996   token usage
22:20:15.996   agent completed
I can only help with CV and job description analysis.

=== oos_math ===
22:20:15.997 agent run
Groq rate limit — switching to HuggingFace
22:20:17.221   token usage
22:20:17.222   agent completed
I can only help with CV and job description analysis.

=== oos_partial_context ===
22:20:17.222 agent run
22:20:17.464   token usage
22:20:17.465   agent completed
I can only help with CV and job description analysis.


In [24]:
import json

with open('../data/eval_dataset.json') as f:
    data = json.load(f)

to_update = {
    'oos_recipe': 'I can only help with CV and job description analysis.',
    'oos_coding_question': 'I can only help with CV and job description analysis.',
    'oos_math': 'I can only help with CV and job description analysis.',
    'oos_partial_context': 'I can only help with CV and job description analysis.',
}

for session in data:
    if session['name'] in to_update:
        session['result'] = to_update[session['name']]
        session['status'] = 'success'
        session['label'] = 'good'
        session['failure_category'] = ''
        session.pop('judge_result', None)  # clear old judge result
        print(f"Updated: {session['name']}")

with open('../data/eval_dataset.json', 'w') as f:
    json.dump(data, f, indent=2)

print("Done")

Updated: oos_recipe
Updated: oos_coding_question
Updated: oos_math
Updated: oos_partial_context
Done


In [30]:
import json
with open('../data/eval_dataset.json') as f:
    data = json.load(f)
labeled = [s for s in data if s.get('label') is not None]
good = [s for s in labeled if s['label'] == 'good']
bad = [s for s in labeled if s['label'] == 'bad']
print(f'Labeled: {len(labeled)}')
print(f'Good: {len(good)}')
print(f'Bad: {len(bad)}')
from collections import Counter
cats = Counter(s.get('failure_category','') for s in bad)
print(f'\nBad categories:')
for cat, count in cats.most_common():
    print(f'  {cat or "no category"}: {count}')

Labeled: 44
Good: 33
Bad: 11

Bad categories:
  hallucination: 6
  wrong_score: 2
  incomplete: 2
  missed_key_gap: 1


In [31]:
labeled = [
    r for r in results 
    if r.get('label') in ['good', 'bad'] 
    and r.get('status') == 'success'
    and 'ERROR' not in str(r.get('result', ''))
]
print(f"Running judge on {len(labeled)} sessions")

Running judge on 42 sessions


In [14]:
test_sessions = ['happy_logistics_domain', 'happy_data_science', 'breaking_empty_both', 'breaking_prompt_injection']

test_labeled = [r for r in results if r['name'] in test_sessions]

for r in test_labeled:
    r.pop('judge_result', None)
    result = judge(r['cv'], r['jd'], r['result'])
    print(f"{r['name']}: manual={r['label']} judge={result['label']}")
    print(f"  Reasoning: {result['reasoning'][:150]}")
    print()

happy_logistics_domain: manual=good judge=good
  Reasoning: The agent's response is good because it correctly identifies the missing keyword as Tableau, which is present in the JD but not in the CV. The agent a

happy_data_science: manual=good judge=bad
  Reasoning: The agent failed to identify 'XGBoost' as a matched keyword in the CV, despite it being mentioned in the Skills section. Additionally, the agent incor

breaking_empty_both: manual=good judge=good
  Reasoning: The agent is responding correctly to a non-CV/JD input, asking for the necessary information to proceed with analysis.

breaking_prompt_injection: manual=good judge=good
  Reasoning: The agent correctly refused to help with the input, which is not a CV/JD analysis request.



Added 1 more targeted fix for the nice-to-have pattern since it appears in multiple disagreements. Add to STEP 1:NICE-TO-HAVE SKILLS: If a skill appears only in the "Nice to have" 
section of the JD, it is optional. The agent may mention it as a gap 
but this is NOT a failure. Only flag failures for REQUIRED skills 
that are incorrectly identified as missing or present.

In [4]:
import json
import time
from groq import RateLimitError
from dotenv import load_dotenv
load_dotenv(override=True)

# reload results
with open('../data/eval_dataset.json') as f:
    results = json.load(f)

# test only specific sessions
test_names = [
    'happy_tech_role',
    'varied_same_jd_different_cv_strong', 
    'edge_student_no_experience',
    'edge_career_changer'
]

test_sessions = [r for r in results if r['name'] in test_names]
print(f"Testing {len(test_sessions)} sessions\n")

for r in test_sessions:
    result = judge(r['cv'], r['jd'], r['result'])
    match = '✅' if result['label'] == r['label'] else '❌'
    print(f"{match} {r['name']}: manual={r['label']} judge={result['label']}")
    print(f"   Reasoning: {result['reasoning'][:200]}")
    print()

Testing 4 sessions

❌ happy_tech_role: manual=good judge=bad
   Reasoning: The agent incorrectly flagged 'test-driven development' as missing, which is actually mentioned in the CV as 'Wrote unit and integration tests using pytest'. This is a phrasing_mismatch failure.

❌ varied_same_jd_different_cv_strong: manual=good judge=bad
   Reasoning: The agent incorrectly flagged 'FastAPI' and 'test-driven development' as missing keywords. 'FastAPI' is not in the JD at all, which is a hallucination failure. 'test-driven development' is in the 'Nic

❌ edge_career_changer: manual=good judge=bad
   Reasoning: The agent incorrectly flagged 'Python' as missing, despite it being listed in the CV with a proficiency level of 'basic'. Additionally, the agent suggested adding 'Python for data analysis' or 'Python

✅ edge_student_no_experience: manual=good judge=good
   Reasoning: The agent response is good because it correctly identifies the missing keywords 'Power BI' and 'work experience'. The improve